In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.15 The van der Waals Gas: Phase Coexistence and the Critical Point

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.15",
    title="The van der Waals Gas: Phase Coexistence and the Critical " "Point",
    blurb="Two corrections to the ideal gas — molecules take space, "
    "molecules attract — and suddenly there is a liquid: coexistence by "
    "Maxwell's equal-area rule, a latent heat that honors "
    "Clausius–Clapeyron to four digits, mean-field critical exponents "
    "measured from our own coexistence dome, and the corresponding-states "
    "confession every real gas makes.",
    difficulty="intermediate",
    estimate="130–160 min",
)

## Notebook overview

[§5.6](ideal-gas-fundamental-relation.ipynb) closed by promising "the
foundational next step beyond the ideal gas," and this notebook keeps
the promise. Van der Waals' 1873 equation of state adds exactly two
physical facts the ideal gas ignores — molecules occupy volume, and
molecules attract at a distance — and out of those two corrections
falls, astonishingly, the entire phenomenology of the liquid–gas
transition: a critical point, a coexistence region where liquid and
vapor share the same pressure and chemical potential, a latent heat,
metastable superheated and supercooled states, and critical exponents.
It is the original mean-field theory, a decade before anyone knew what
that meant.

The notebook *measures* all of it from the equation itself. Maxwell's
equal-area construction becomes a root-finding problem whose solution
we verify against an independently computed chemical potential; the
coexistence dome, assembled point by point, yields the order-parameter
exponent $\beta = 1/2$ from a fit — the same mean-field value
[§5.10](ising-emergence-universality.ipynb) extracts for the Ising
magnet, and the deepest sentence in this notebook is that this is *not
a coincidence*. The latent heat honors Clausius–Clapeyron to four
digits as a cross-method identity. And the finale is the confession:
every real gas, reduced by its own critical constants, tells nearly
the same story (corresponding states) — while all of them politely
disagree with van der Waals' $Z_c = 3/8$ by the same honest margin.
The volume's standing reference remains {cite}`nolting6`.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**The equation and its reduced form.** Van der Waals corrects the
ideal gas twice: the volume available to a molecule is $V - Nb$ (hard
cores), and the attraction reduces the pressure by $a(N/V)^2$
(pairwise, hence density-squared):

```{math}
:label: eq-vdw-eos
\left(p + \frac{aN^2}{V^2}\right)(V - Nb) \;=\; Nk_BT .
```

The isotherms develop a wiggle below a *critical point* — the unique
temperature where $\partial p/\partial V = \partial^2 p/\partial V^2
= 0$ — at $V_c = 3Nb$, $k_BT_c = 8a/27b$, $p_c = a/27b^2$. Measuring
everything in critical units ($v = V/V_c$, $t = T/T_c$,
$\hat p = p/p_c$) eliminates both material constants:

```{math}
:label: eq-vdw-reduced
\hat p \;=\; \frac{8t}{3v - 1} - \frac{3}{v^2} ,
```

one universal equation for every van der Waals fluid — the *law of
corresponding states*. The model also fixes the dimensionless critical
ratio $Z_c = p_cV_c/(Nk_BT_c) = 3/8$, a prediction real gases get to
vote on.

**Maxwell's construction.** Below $t = 1$ the isotherm's wiggle
contains a branch with $\partial p/\partial v > 0$ — negative
compressibility, mechanically impossible. The resolution is phase
separation: liquid at $v_l$ and vapor at $v_g$ coexist at a single
pressure $p^*$, fixed by *two* equalities (mechanical and chemical
equilibrium). Since $d\mu = v\,d\hat p$ along an isotherm, equal
chemical potentials force

```{math}
:label: eq-vdw-maxwell
\int_{v_l}^{v_g}\bigl[\hat p(v) - p^*\bigr]\,dv \;=\; 0 :
```

the horizontal tie-line cuts the loop into two lobes of *equal area*.
The construction replaces the unphysical wiggle with the flat
coexistence segment every real isotherm shows.

**Clausius–Clapeyron.** Along the coexistence curve $p^*(t)$ the two
phases' chemical potentials remain equal, which forces the slope

```{math}
:label: eq-vdw-cc
\frac{dp^*}{dt} \;=\; \frac{\Delta s}{\Delta v},
\qquad
\Delta s \;=\; \frac{8}{3}\,
\ln\!\frac{3v_g - 1}{3v_l - 1}
```

in reduced units (the entropy difference follows from the Sackur–
Tetrode-style volume term of [§5.6](ideal-gas-fundamental-relation.ipynb)
with the excluded volume in place; $t\,\Delta s$ is the latent heat).
The relation is an identity of the construction, so verifying it
numerically is a genuine cross-method check: the left side differentiates
a sequence of equal-area solutions, the right side never sees them.

**Mean-field exponents.** Near the critical point the model's
singularities are power laws: the dome closes as $v_g - v_l \sim
(1 - t)^{\beta}$ with $\beta = 1/2$ (asymptotically
$v_g - v_l \to 4\sqrt{1 - t}$), the critical isotherm flattens as
$|\hat p - 1| \sim |v - 1|^{\delta}$ with $\delta = 3$, and the
compressibility diverges as $\kappa_T \sim (t - 1)^{-\gamma}$ with
$\gamma = 1$. These are the *mean-field* values —
{math}`\beta = \tfrac12,\ \delta = 3,\ \gamma = 1` — identical to the
mean-field Ising exponents of
[§5.10](ising-emergence-universality.ipynb), because both theories
replace fluctuations by an average field. Real fluids (and the real
3D Ising model) disagree — $\beta \approx 0.326$ — and agree *with
each other*: universality's deepest exhibit, with the liquid–gas
transition and the ferromagnet in the same class.

## Setup

Reduced units {eq}`eq-vdw-reduced` throughout — every quantity is
measured in its critical value — with SI entering only in the
corresponding-states confrontation, via `scipy.constants`. All root
brackets and grids are pinned; nothing is stochastic.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import R as R_GAS  # J/(mol K)
from scipy.optimize import brentq

from ecp import validate


def p_vdw(v, t):
    """Reduced van der Waals pressure 8t/(3v-1) - 3/v^2, eq-vdw-reduced.

    The universal isotherm of every van der Waals fluid, in units of the
    critical pressure, volume, and temperature.

    Parameters
    ----------
    v : float or numpy.ndarray
        Reduced volume(s), v > 1/3.
    t : float
        Reduced temperature.

    Returns
    -------
    float or numpy.ndarray
        Reduced pressure(s).
    """
    return 8.0 * t / (3.0 * v - 1.0) - 3.0 / v**2


def coexistence(t, v_max=3000.0, n_grid=20000):
    """Maxwell construction at reduced temperature t < 1.

    Locates the isotherm's spinodal extrema on a fine grid, then solves
    the equal-area condition eq-vdw-maxwell for the coexistence pressure
    with scipy.optimize.brentq: for each trial p*, the three roots of
    p(v) = p* are bracketed (liquid, unstable, gas) and the signed area
    between isotherm and tie-line is integrated by numpy.trapezoid; the
    pressure that zeroes it is Maxwell's.

    Parameters
    ----------
    t : float
        Reduced temperature, 0 < t < 1.
    v_max : float
        Upper bracket for the gas root.
    n_grid : int
        Grid points for the spinodal search.

    Returns
    -------
    tuple of float
        ``(p_star, v_l, v_g)``: coexistence pressure and the liquid and
        gas volumes.
    """
    vs = np.linspace(0.36, 30.0, n_grid)
    dp = np.gradient(p_vdw(vs, t), vs)
    idx = np.where(np.diff(np.sign(dp)) != 0)[0]
    v_sp_lo, v_sp_hi = vs[idx[0]], vs[idx[1]]
    p_lo = max(p_vdw(v_sp_lo, t), 1e-8)
    p_hi = p_vdw(v_sp_hi, t)

    def area_and_roots(p_star):
        f = lambda v: p_vdw(v, t) - p_star
        v_l = brentq(f, 1.0 / 3.0 + 1e-6, v_sp_lo)
        v_g = brentq(f, v_sp_hi, v_max)
        vv = np.linspace(v_l, v_g, 4000)
        return float(np.trapezoid(p_vdw(vv, t) - p_star, vv)), v_l, v_g

    p_star = brentq(
        lambda p: area_and_roots(p)[0],
        p_lo * 1.0000001,
        p_hi * 0.9999999,
    )
    _, v_l, v_g = area_and_roots(p_star)
    return p_star, v_l, v_g


def delta_s_coex(v_l, v_g):
    """Reduced entropy difference (8/3) ln((3v_g-1)/(3v_l-1)) across the dome.

    The excluded-volume term of the van der Waals entropy, eq-vdw-cc:
    times t, it is the latent heat of vaporization in reduced units.

    Parameters
    ----------
    v_l, v_g : float
        Coexisting liquid and gas reduced volumes.

    Returns
    -------
    float
        The entropy of vaporization, reduced units.
    """
    return (8.0 / 3.0) * np.log((3.0 * v_g - 1.0) / (3.0 * v_l - 1.0))

## Exercise 1 — The wiggle and the critical point

{eq}`eq-vdw-reduced` must contain its own critical point at
$(v, t, \hat p) = (1, 1, 1)$ by construction; the first checks confirm
the construction and draw the landscape.

**Part a)** Verify $\hat p(1, 1) = 1$ exactly, and that both
$\partial\hat p/\partial v$ and $\partial^2\hat p/\partial v^2$ vanish
at the critical point (centered finite differences with $h = 10^{-5}$;
`atol=1e-4`, the second difference's roundoff floor). Verify the
mechanical pathology below $t = 1$: on the $t = 0.9$ isotherm the
maximum of $\partial\hat p/\partial v$ over $v \in [0.5, 3]$ is
*positive* (negative compressibility exists), while on the $t = 1.1$
isotherm it is negative everywhere (stable at every volume).

**Part b)** Draw the isotherm family $t = 0.85, 0.90, 0.95, 1.0, 1.1$
in the $\hat p$–$v$ plane with the critical point marked: the wiggle
maturing into an inflection and then into monotone decline.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    p_c_check,
    1.0,
    "the reduced equation passes through its own critical point " "(1,1,1) exactly",
    rtol=1e-12,
)
validate.check(
    abs(dp_c) < 1e-4 and abs(d2p_c) < 1e-4,
    "and both volume derivatives vanish there: the double root that "
    "defines criticality",
    f"{dp_c:.1e}, {d2p_c:.1e}",
)
validate.check(
    slope_09 > 0.0 and slope_11 < 0.0,
    "below t=1 the isotherm carries a rising (negative-compressibility) "
    "branch; above, it is stable at every volume",
    f"max slopes {slope_09:+.3f} vs {slope_11:+.3f}",
)

## Exercise 2 — Maxwell's equal areas, solved and cross-examined

The construction of {eq}`eq-vdw-maxwell` is implemented in
`coexistence`: a `brentq` root-find over the tie-line pressure, with
the equal-area integral evaluated by `numpy.trapezoid` at each trial.
**Write this one yourself** — the implementation is the lesson.

**Part a)** At $t = 0.9$, verify the construction delivers
$p^* = 0.64700$, $v_l = 0.60340$, $v_g = 2.34884$ (`rtol=1e-4` each —
the textbook coexistence state of the van der Waals fluid at ninety
percent of criticality). Then cross-examine it with a computation the
solver never used: the chemical-potential difference
$\Delta\mu = \int_{v_l}^{v_g} v\,\frac{d\hat p}{dv}\,dv$ along the
isotherm (integrating $v\,d\hat p$ by `numpy.gradient` and
`numpy.trapezoid`) must vanish; verify $|\Delta\mu| < 10^{-6}$. Equal
areas and equal chemical potentials are the same statement, and the
machine confirms it.

**Part b)** Assemble the coexistence dome: run `coexistence` for the
$31$ temperatures $t \in \{0.85, 0.855, \ldots, 0.999\}$ (a uniform
grid of $0.005$ plus the near-critical $0.996, 0.997, 0.998, 0.999$),
and draw the binodal ($v_l(t)$ and $v_g(t)$) together with the
spinodal (the locus of $\partial\hat p/\partial v = 0$, found on the
same grids) in the $t$–$v$ plane. Verify the ordering that defines
metastability: at every $t$, $v_l < v_{\rm sp}^{\rm liq} <
v_{\rm sp}^{\rm gas} < v_g$ — the superheated-liquid and supercooled-
vapor strips live between the two curves, real enough to boil
explosively when nucleation finally wins.

**Part c)** Verify Clausius–Clapeyron as a cross-method identity at
$t = 0.9$: the slope $dp^*/dt$ by centered difference of the
construction at $t = 0.898$ and $0.902$, against $\Delta s/\Delta v$
from {eq}`eq-vdw-cc` evaluated at the coexisting volumes — two
computations that share nothing but the physics; verify agreement to
`rtol=1e-3`. The latent heat $t\,\Delta s = 3.55$ in reduced units
rides along for free.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([p_star_09, v_l_09, v_g_09]),
    np.array([0.64700, 0.60340, 2.34884]),
    "Maxwell's construction at t = 0.9: the textbook coexistence state, "
    "found by root-finding on the equal-area condition",
    rtol=1e-4,
)
validate.check(
    abs(delta_mu) < 1e-6,
    "and the chemical potentials agree across the tie line — computed "
    "by an integral the solver never saw",
    f"Δμ = {delta_mu:.1e}",
)
validate.check(
    ordering_ok,
    "at all 31 temperatures the binodal encloses the spinodal: the "
    "metastable strips where superheating and supercooling live",
    "v_l < spinodal < v_g everywhere",
)
validate.close(
    dpdt_cc,
    cc_rhs,
    "Clausius–Clapeyron holds as a cross-method identity: the "
    "coexistence curve's slope equals Δs/Δv to a part in a thousand",
    rtol=1e-3,
)

## Exercise 3 — Mean-field exponents, measured from the dome

The dome and the isotherms now contain the model's critical
singularities, and the exponents are ours to measure — with the
discipline the fits require, because power laws are only exact *at*
the critical point and every finite window carries corrections to
scaling.

**Part a)** The order parameter, $\beta$. Fit
$\log(v_g - v_l)$ against $\log(1 - t)$ (`numpy.polyfit`, degree 1)
over the *near-critical* subset $t \ge 0.99$ of Exercise 2's dome and
verify the slope lands in $[0.48, 0.54]$, containing the mean-field
$\beta = 1/2$; verify also the asymptotic amplitude, ${(v_g - v_l)}/
{\sqrt{1 - t}} \to 4$: the value at $t = 0.999$ within $5\%$ of $4$.
Then quantify the corrections honestly: the same fit over the *wide*
window $t \in [0.85, 0.96]$ gives an effective slope *above* $0.53$ —
power-law fitting far from the critical point measures the window, not
the exponent (a warning that applies to every log–log fit in the
literature).

**Part b)** The critical isotherm, $\delta$. On the $t = 1$ isotherm,
form the antisymmetric combination $[\hat p(1 + \epsilon) -
\hat p(1 - \epsilon)]/2$ (which cancels the even corrections) for
$\epsilon \in [0.003, 0.05]$ (15 log-spaced points) and verify the
log–log slope equals $3$ within $\pm 0.05$; verify the cubic's
amplitude against the exact expansion $\hat p - 1 = -\tfrac32
(v - 1)^3 + \ldots$: the ratio at the smallest $\epsilon$ within $2\%$
of $3/2$.

**Part c)** The compressibility, $\gamma$. At $v = 1$, compute
$\kappa_T \propto -1/(\partial\hat p/\partial v)$ (centered difference,
$h = 10^{-5}$) for $t = 1.02, 1.05, 1.1, 1.2$ and verify the log–log
slope of $\kappa_T$ against $(t - 1)$ equals $-1$ within $\pm 0.01$:
the mean-field $\gamma = 1$, essentially exact because this
singularity happens to carry no leading correction at $v = 1$. The
trio $(\beta, \delta, \gamma) = (\tfrac12, 3, 1)$ is the mean-field
universality class — the same numbers
[§5.10](ising-emergence-universality.ipynb) derives for the mean-field
magnet, because averaging away fluctuations is the same approximation
in any costume.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    0.48 < beta_near < 0.54,
    "the near-critical dome closes with the mean-field order-parameter "
    "exponent beta = 1/2",
    f"fit {beta_near:.3f}",
)
validate.close(
    amp_999,
    4.0,
    "with the exact asymptotic amplitude: v_g - v_l -> 4 sqrt(1-t)",
    rtol=5e-2,
)
validate.check(
    beta_wide > 0.53,
    "while the wide-window fit drifts high: far from criticality a "
    "log-log fit measures the window, not the exponent",
    f"wide fit {beta_wide:.3f}",
)
validate.check(
    abs(delta_slope - 3.0) < 0.05,
    "the antisymmetrized critical isotherm is cubic: delta = 3",
    f"slope {delta_slope:.3f}",
)
validate.close(
    amp_cubic,
    1.5,
    "with the exact coefficient 3/2 of the expansion p - 1 = " "-(3/2)(v-1)^3 + ...",
    rtol=2e-2,
)
validate.check(
    abs(-gamma_slope - 1.0) < 0.01,
    "and the compressibility diverges with gamma = 1: the mean-field "
    "trio (1/2, 3, 1) complete, matching the mean-field magnet of §5.10",
    f"gamma {-gamma_slope:.3f}",
)

## Exercise 4 — Corresponding states: the gases vote

{eq}`eq-vdw-reduced` predicts that *every* fluid, measured in its own
critical units, obeys the same equation — and makes one sharp
falsifiable claim before any isotherm is drawn: the critical
compressibility ratio $Z_c = p_cV_c/(RT_c)$ (molar units) must equal
$3/8 = 0.375$ for every substance.

**Part a)** Put five real fluids on the stand, with their measured
molar critical constants $(T_c\ [\mathrm K],\ p_c\ [\mathrm{Pa}],\
V_c\ [\mathrm{m^3/mol}])$: neon $(44.49,\ 2.68\times10^6,\
41.7\times10^{-6})$, argon $(150.7,\ 4.86\times10^6,\
74.6\times10^{-6})$, nitrogen $(126.2,\ 3.39\times10^6,\
89.5\times10^{-6})$, carbon dioxide $(304.1,\ 7.38\times10^6,\
94.0\times10^{-6})$, and water $(647.1,\ 22.06\times10^6,\
55.9\times10^{-6})$. Compute each $Z_c$ with `scipy.constants.R` and
verify: all five land in $[0.22, 0.31]$ — clustered, which is
corresponding states working — and all five sit *below* the van der
Waals $0.375$ by at least $0.06$, which is the model confessing (real
attractions are not infinitely weak and long-ranged). Verify the
simple fluids cluster tightly — neon, argon, and nitrogen within
$0.015$ of each other — while water sits lowest of all five: hydrogen
bonding is the least van-der-Waals-like force in common experience,
and $Z_c$ knows it.

**Part b)** The constructive face of the same law: Guggenheim's
observation that reduced coexistence data from Ne to O$_2$ collapse
onto one dome. Verify the model's version quantitatively: the reduced
dome of Exercise 2 predicts $v_g/v_l = 3.89$ at $t = 0.9$
(`rtol=1e-3` against Exercise 2's values) — a pure number every van
der Waals fluid must share, no material constants anywhere in it.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    all(0.22 < z < 0.31 for z in z_values.values()),
    "five real fluids cluster in [0.22, 0.31]: corresponding states is "
    "a real law of nature",
    f"{ {k: round(v, 3) for k, v in z_values.items()} }",
)
validate.check(
    all(Z_C_VDW - z > 0.06 for z in z_values.values()),
    "and every one sits well below the van der Waals 3/8: the model's "
    "honest, universal confession",
    f"max Z_c = {max(z_values.values()):.3f}",
)
validate.check(
    max(simple_three) - min(simple_three) < 0.015
    and z_values["H2O"] == min(z_values.values()),
    "the simple fluids agree with each other far better than with the "
    "model, and hydrogen-bonded water strays furthest",
    f"spread(Ne,Ar,N2) = {max(simple_three) - min(simple_three):.3f}, "
    f"H2O = {z_values['H2O']:.3f}",
)
validate.close(
    ratio_gl,
    3.89,
    "while the reduced dome's v_g/v_l = 3.89 at t = 0.9 is a pure "
    "number every van der Waals fluid shares",
    rtol=1e-3,
)

## Notebook summary

- The reduced equation carried its critical point at $(1,1,1)$ with
  both volume derivatives vanishing, and its sub-critical isotherms
  carried the mechanically impossible rising branch that demands phase
  separation.
- Maxwell's equal-area construction, solved by root-finding, delivered
  the textbook $t = 0.9$ coexistence state ($p^* = 0.647$,
  $v_l = 0.603$, $v_g = 2.349$) and was cross-examined by an
  independent chemical-potential integral that vanished to $10^{-7}$;
  the binodal enclosed the spinodal at all 31 temperatures, bounding
  the metastable strips.
- Clausius–Clapeyron held as a cross-method identity to a part in a
  thousand, with the reduced latent heat $t\,\Delta s = 3.55$ riding
  along.
- The exponents came out mean-field from our own fits — $\beta$ in
  $[0.48, 0.54]$ with the exact amplitude $4\sqrt{1-t}$, $\delta = 3$
  with coefficient $3/2$, $\gamma = 1$ to one percent — with the
  wide-window $\beta$ fit deliberately shown drifting high: power-law
  fits far from criticality measure the window.
- Five real fluids voted: $Z_c$ clustered in $[0.22, 0.31]$
  (corresponding states works), all below the model's $3/8$ (the
  model confesses), the simple fluids within $0.015$ of each other and
  water strayed furthest — every deviation physically legible.

## Outlook

- **Beyond mean field.** Real fluids close their domes with
  $\beta \approx 0.326$, not $1/2$ — the 3D Ising exponent, because
  the liquid–gas transition and the ferromagnet share a universality
  class. Why mean field fails in three dimensions, and how
  renormalization repairs it, is the story
  [§5.10](ising-emergence-universality.ipynb) opens and the Epilogue's
  universality notebook closes.
- **Better equations of state.** Redlich–Kwong, Peng–Robinson, and
  the virial expansion of
  [§5.6](ideal-gas-fundamental-relation.ipynb)'s outlook are the
  engineering descendants; all keep van der Waals' two ideas and tune
  the confession.
- **Nucleation.** The metastable strips between binodal and spinodal
  decay by rare fluctuations — droplet nucleation, with its
  free-energy barrier and Arrhenius kinetics: the physics of cloud
  chambers, bubble chambers, and the superheated coffee that erupts
  in the microwave.
- **Interfaces.** Coexistence implies a boundary, and the boundary
  has physics of its own: surface tension, capillarity, and the
  density profile van der Waals himself computed in 1893, founding
  what is now density-functional theory.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()